# Meqpy Tutorial — 5a. Cubes and Transitions

[← Previous: 4. BandSystem and BandTransitions](04_BandSystem_BandTransitions.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 5b. Molecule →](05b_Molecule.ipynb)

In [ ]:
import meqpy

import numpy as np
import matplotlib.pyplot as plt

### Overview

- [5. Spatial Resolution](#spatial)
    - [5.1 Cube class](#spatial_cube)
        - [5.1.1 Cubes from file](#spatial_cube_file)
        - [5.1.2 Cubes from 2p<sub>z</sub> vector](#spatial_cube_2pz)
    - [5.2 Transition Class](#spatial_transition)
        - [5.2.1 Dyson](#spatial_transition_dyson)
            - [Coordinates and Slice Height](#spatial_transition_dyson_coords)
            - [Properties and Methods](#spatial_transition_dyson_methods)
    - 5.3 Molecule [→ 5b Molecule](05b_Molecule.ipynb)
        - 5.3.1 Charging Transitions: Dyson
            - Add Dyson Transition
            - Missing Dyson Transitions
            - Molecule and Dyson Shape
            - Dyson Amplitudes
        - 5.3.2 Helper Functions
        - 5.3.3 Charging Rates
            - Coupling to plane wave Sample
            - Coupling to *s*-wave Tip
            - Point Spectroscopy
    - 5.4 Example Experiments [→ 5c Molecule Examples](05c_Molecule_Examples.ipynb)
        - 5.4.1 Constant Height Map
        - 5.4.2 I(V) Point Spectroscopy


<a id='spatial'></a>
## 5. Spatial Resolution
Another extension of the ``System`` class is the ``Molecule`` class, which allows systems to be solved with spatial resolution. For this, molecular orbitals can be used to determine the spatial dependence of various transition rates — for example, Dyson orbitals for the charge transitions. These orbitals must be computed by external means, e.g. using DFT or tight-binding methods, and then loaded into ``meqpy`` via the ``Cube`` class. A ``Cube`` object can then be used to instantiate a ``Transition`` object, such as ``Dyson``, which handles the charging transition rates in a ``Molecule``.

In this chapter, we first discuss the ``Cube`` class and the ``Transition`` and ``Dyson`` classes, before looking at the ``Molecule`` class and some example experiments.

<a id='spatial_cube'></a>
### 5.1 Cube
The role of the ``Cube`` class is to provide a convenient way to handle volumetric data. Unless stated otherwise, all coordinates are given with respect to the origin of the volumetric data and are in units of Ångstrom.

Properties:
- ``atoms``: ``ase.Atoms`` object containing all atoms
- ``original_atoms``: same as ``atoms``, but in original Cartesian coordinates
- ``elements``: list of element symbols of all atoms
- ``masses``: masses of atoms in atomic units
- ``cart_coords``: coordinates of all atoms
- ``origin``: origin of volumetric data, in original Cartesian coordinates
- ``spacing``: (3,3) matrix representing the voxel dimensions
- ``center_of_mass``: center of mass
- ``data``: 3D array containing the volumetric data
- ``voxel_size``: volume of single voxel in volumetric data
- ``magsqr``: integral over the volumetric data squared

In addition, there are two methods:
- ``get_axis_grid()``: returns the axis grid in internal coordinates
- ``get_slice_data()``: returns a 2D slice of the volumetric data along a given axis

There are two ways to create a cube object: from file ([5.1.1](#spatial_cube_file)) or from tight-binding based calculations ([5.1.2](#spatial_cube_2pz)).

<a id='spatial_cube_file'></a>
#### 5.1.1 Cube from file
A ``Cube`` object can simply be created by calling ``meqpy.Cube`` and providing the path to a Gaussian-formatted cube file. It uses the ``ase.io.cube.read_cube`` method from the ``ase`` package for loading and parsing the cube file.



In [ ]:
# loading two cube files from DFT calculations
# for the HOMO and the LUMO of naphthalene
homo_cub = meqpy.Cube("./tutorial_files/naphthalene_homo.cub")
lumo_cub = meqpy.Cube("./tutorial_files/naphthalene_lumo.cub")

In [ ]:
# return grid values along first axis
# always starts at 0 and stepsizes are in Ångstrom
homo_cub.get_axis_grid(0)

# get_axis_grid() can be used for non-cartesian cubes
# e.g. with rhombic cells

In [ ]:
# get slice of cube data 1.5Å above the center of mass

distance = 1.5  # Å
distance += homo_cub.center_of_mass[2]

slice_data = homo_cub.get_slice_data(distance)

slice_data.shape

In [ ]:
# Plot slice wave function of HOMO and LUMO orbitals
# taken 1.5 Å above the center of mass
rel_slice_height = 1.5  # Å

fig, ax = plt.subplots(1, 2, figsize=(5, 3))

for i, cub in enumerate([homo_cub, lumo_cub]):
    # get x, y grid
    x, y = cub.get_axis_grid(0), cub.get_axis_grid(1)

    # get slice of data
    slice_height = rel_slice_height + cub.center_of_mass[2]
    slice_data = cub.get_slice_data(slice_height)

    vrange = np.max(np.abs(slice_data))
    ax[i].pcolormesh(y, x, slice_data, cmap="bwr", vmin=-vrange, vmax=vrange)

    # make mask to ignore hydrogen
    c_mask = cub.atoms.numbers > 1
    sites = cub.cart_coords[c_mask]
    ax[i].plot(sites[:, 1], sites[:, 0], "o", color="k", markerfacecolor="None")

    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")

ax[0].set_title("HOMO from cube file")
ax[1].set_title("LUMO from cube file")
plt.tight_layout(pad=2)
plt.show()

<a id='spatial_cube_2pz'></a>
#### 5.1.2 Cube_2pz class
In addition to Gaussian-formatted cube files, ``meqpy`` also supports results from tight-binding calculations that use $2p_z$ orbitals as a basis, via the ``Cube_2pz`` class. The class inherits from ``Cube`` and takes the following input:

Required:
- ``positions``: list of Cartesian coordinates for each site (i.e. each $2p_z$ orbital)
- ``eigenvector``: 1D array containing the real coefficients of the orbital for each site

Optional:
- ``boundary``: boundary around atoms in Ångstrom used to determine the cube size (default: 5Å)
- ``spacing``: size of cubic voxels in Ångstrom (default: 0.333333 Bohr)
- ``elements``: list of elements corresponding to each site; if ``None`` (default), all sites are assumed to be carbon
- ``fill_cube``: if ``True``, the wave function is calculated for the full volumetric data

In the following we will be using the results from Mean-Field Hubbard calculations of a naphthalene molecule.

In [ ]:
sites = np.array(
    [
        [8.60, 12.78, 0.00],
        [11.06, 12.78, 0.00],
        [7.37, 13.49, 0.00],
        [9.83, 13.49, 0.00],
        [12.29, 13.49, 0.00],
        [7.37, 14.91, 0.00],
        [9.83, 14.91, 0.00],
        [12.29, 14.91, 0.00],
        [8.60, 15.62, 0.00],
        [11.06, 15.62, 0.00],
    ]
)

homo_vec = np.array(
    [
        +0.41988137,
        -0.41988063,
        +0.27148131,
        +0.00000000,
        -0.27148104,
        -0.27149031,
        +0.00000000,
        +0.27148989,
        -0.41987062,
        +0.41986996,
    ]
)

lumo_vec = np.array(
    [
        -0.41987029,
        +0.41986972,
        +0.27148989,
        +0.00000000,
        -0.27148983,
        +0.27148175,
        +0.00000000,
        -0.27148122,
        -0.41988152,
        +0.41988102,
    ]
)

In [ ]:
# create a Cube_2pz object from MFH results
homo_2pz = meqpy.Cube_2pz(positions=sites, eigenvector=homo_vec)
lumo_2pz = meqpy.Cube_2pz(positions=sites, eigenvector=lumo_vec)

In [ ]:
# Plot slice wave function of HOMO and LUMO orbitals
# taken 1.5 Å above the center of mass
rel_slice_height = 1.5  # Å

fig, ax = plt.subplots(1, 2, figsize=(5, 3))

for i, cub in enumerate([homo_2pz, lumo_2pz]):
    # get x, y grid
    x, y = cub.get_axis_grid(0), cub.get_axis_grid(1)

    # get slice of data
    slice_height = rel_slice_height + cub.center_of_mass[2]
    slice_data = cub.get_slice_data(slice_height)

    vrange = np.max(np.abs(slice_data))
    ax[i].pcolormesh(y, x, slice_data, cmap="bwr", vmin=-vrange, vmax=vrange)

    ax[i].plot(
        homo_2pz.cart_coords[:, 1],
        homo_2pz.cart_coords[:, 0],
        "o",
        color="k",
        markerfacecolor="None",
    )

    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")

ax[0].set_title("HOMO from 2$p_z$")
ax[1].set_title("LUMO from 2$p_z$")
plt.tight_layout(pad=2)
plt.show()

Note: The ``Cube_2pz`` object does not calculate the wave function for the whole volumetric data by default, as it is not necessarily needed:
- ``Cube_2pz.magsqr`` is calculated using the eigenvector instead of the data array
- ``Cube_2pz.get_slice_data()`` calculates the required slice on demand

The ``Cube_2pz.data`` array will be created upon initialization, but is filled with zeros.

To fill the volumetric data, either choose ``fill_cube = True`` upon initialization or use ``Cube_2pz.fill_cube()``.

In [ ]:
# initialize Cube_2pz but do not fill data array
homo_2pz = meqpy.Cube_2pz(positions=sites, eigenvector=homo_vec, fill_cube=False)
print("Without filling data array:")

print(f"Shape of data: {homo_2pz.data.shape}")
print(f"Cube.magsqr(): {homo_2pz.magsqr: .03f}")


integrated_data = np.sum(homo_2pz.data**2) / meqpy.constants.BOHR**3
integrated_data *= homo_2pz.voxel_size
print(f"Integrated Data: {integrated_data: .03f}")

In [ ]:
print("After filling data array:")

homo_2pz.fill_cube()

integrated_data = np.sum(homo_2pz.data**2) / meqpy.constants.BOHR**3
integrated_data *= homo_2pz.voxel_size
print(f"Integrated Data: {integrated_data: .03f}")

<a id='spatial_transition'></a>
### 5.2 Transition Classes

The ``Transition`` class is an abstract parent class from which other spatial transition classes inherit. It provides the essential tools shared by all spatial transition classes:
- initialization can be done by providing a ``Cube`` object or a path to a Gaussian-formatted cube file
- it validates that the z-axis of the provided cube is aligned with the z-axis of the coordinate system and orthogonal to the other two axes
- if ``center_mass = True`` (default), the origin of the coordinate system is shifted to the center of mass of the molecule

Properties:
- ``spacing``: (3,3) matrix representing the voxel dimensions
- ``origin``: origin of volumetric data, depends on ``center_mass`` value during instantiation
- ``steps``: step sizes of the axis grids

Methods:
- ``grid(pad)``: list containing the axis grid of all axes, padded by ``pad`` points on each side
- ``get_cart_axis(axis)``: returns the Cartesian coordinates of the given axis, for a cube with orthogonal axes aligned with the coordinate system
- ``mesh_cartesian(pad)``: returns a meshgrid of the data in Cartesian coordinates, padded by ``pad`` points on each side

<a id='spatial_transition_dyson'></a>
#### 5.2.1 Dyson
The ``Dyson`` class is an extension of the ``Transition`` class and is used for handling charging transitions within the ``Molecule`` class. To do this, it stores one slice of the wave function from a ``Cube`` object and extrapolates the wave function exponentially into the vacuum up to a given height. The square of the wave function at a given point is proportional to the charging rate.

<a id='spatial_transition_dyson_coords'></a>
**Coordinates and Slice Height**

Since ``Dyson`` inherits from ``Transition``, it can be instantiated as well using either a ``Cube`` object or a path to a cube file. In addition, it takes two optional parameters:
- ``slice_height``: z-value at which to take the slice, default is 1.5Å
- ``center_mass``: shift the origin of the coordinate system to the center of mass of the molecule, default is ``True``

Note: ``center_mass`` is applied before ``slice_height``, so if ``center_mass = True``, ``slice_height`` corresponds to the vertical distance between the slice and the center of mass.

In [ ]:
# instantiate Dyson via Cube object
lumo_cub = meqpy.Cube("./tutorial_files/naphthalene_lumo.cub")
lumo_dyson = meqpy.Dyson(lumo_cub)
lumo_dyson

In [ ]:
# instantiate Dyosn directly from file
lumo_dyson = meqpy.Dyson("./tutorial_files/naphthalene_lumo.cub")
lumo_dyson

<a id='spatial_transition_dyson_methods'></a>
The ``Dyson`` class has additional and modified properties and methods compared to ``Transition``:

Properties:
- ``data``: 2D array containing the slice of the wave function
- ``slice_height``: z-value corresponding to the data slice
- ``shape``: shape of the stored data slice
- ``spacing``: (2,2) matrix representing the pixel dimensions
- ``amplitude``: magnitude of the Dyson orbital, obtained from ``Cube.magsqr``
- ``x`` and ``y``: Cartesian coordinates of the x and y axes, if they are orthogonal and aligned with the coordinate system

Methods:
- ``extrapolate_wavefunction``: returns the wave function extrapolated exponentially into the vacuum
- ``coupling_strength``: returns the square of ``extrapolate_wavefunction``

The ``extrapolate_wavefunction`` method takes three parameters:
- ``height``: height up to which the wave function is extrapolated
- ``kappa``: decay constant to use for the extrapolation
- ``pad``: pads the data slice on all sides with ``pad`` points, default is 0

``height`` can be a float or a 1D array, whereas ``kappa`` can be a float or an array of arbitrary shape. The output shape of the method is ``Dyson.shape + height.shape + kappa.shape``. If ``pad > 0``, ``Dyson.shape`` is increased by ``2*pad`` in both dimensions.

In [ ]:
# one height, one decay constant, no padding
lumo_dyson.extrapolate_wavefunction(height=5.0, kappa=1.15).shape

In [ ]:
# one height, one decay constant, with padding
lumo_dyson.extrapolate_wavefunction(5.0, 1.15, pad=10).shape

In [ ]:
# one heigh, various kappa
kappa = np.ones((5, 3, 3)) * 1.15
lumo_dyson.extrapolate_wavefunction(5.0, kappa).shape

In [ ]:
# various heights, single kappa
height = np.arange(2.0, 8.0)
height_slices = lumo_dyson.extrapolate_wavefunction(height, 1.15)
height_slices.shape

In [ ]:
# Plot slice of wave function of LUMO for various heights

n = height_slices.shape[-2]

fig, ax = plt.subplots(1, n, figsize=(10, 3))

for i in range(n):
    islice = height_slices[..., i, 0]
    vrange = np.max(np.abs(islice))
    ax[i].pcolormesh(
        lumo_dyson.y, lumo_dyson.x, islice, cmap="bwr", vmin=-vrange, vmax=vrange
    )

    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")
    ax[i].set_title(f"height: {height[i]:.01f} Å")

plt.tight_layout(pad=1)
plt.show()

The ``Dyson.coupling_strength`` method is a simple wrapper to return the square of ``Dyson.extrapolate_wavefunction`` and takes the same inputs. In addition, the optional parameter ``squeeze`` (default: ``True``) will squeeze out any dimensions of length 1 from the returned array.

In [ ]:
# various heights, single kappa
height = np.arange(2.0, 8.0)
height_slices2 = lumo_dyson.coupling_strength(height, 1.15, squeeze=True)

# the kappa shape of length 1 is squeezed out
height_slices2.shape

In [ ]:
# Plot slice of wave function of LUMO for various heights

n = height_slices2.shape[-1]

fig, ax = plt.subplots(1, n, figsize=(10, 3))

for i in range(n):
    islice = height_slices2[..., i]
    ax[i].pcolormesh(lumo_dyson.y, lumo_dyson.x, islice, cmap="grey")

    ax[i].set_xticks([])
    ax[i].set_yticks([])
    ax[i].axes.set_aspect("equal")
    ax[i].set_title(f"height: {height[i]:.01f} Å")

plt.tight_layout(pad=1)
plt.show()

---

[← Previous: 4. BandSystem and BandTransitions](04_BandSystem_BandTransitions.ipynb) | [🏠 Index](00_Overview.ipynb) | [Next: 5b. Molecule →](05b_Molecule.ipynb)